In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rdkit==2024.9.5
!pip install torch_geometric==2.5.3

In [ ]:
import os
import sys
import torch
import time
from tqdm import tqdm
import datetime
doc_name = "/content/drive/MyDrive/HeckLit-Code"
sys.path.append(doc_name)
from utils.rxn import *
from utils.molecule import *
from utils.dataset_analysis import *
from models.DeepLearnModel import *
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
rs_list = [1,2,3,4,5]
eval_metrics = np.zeros((1+len(rs_list), 6))
columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
index = []
for rs in rs_list:
  index.append("%s" % rs)
index.append("avg±std")
eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

In [ ]:
for rs in rs_list:
  # 1. import data
  data = pd.read_excel("%s/data/Heck/JCP raw data.xlsx" % doc_name)
  random_state = rs
  data = data.sample(random_state=random_state, frac=1).reset_index(drop=True)

  # 2. build dataset & dataloader
  feat_set = list()
  y_set = list()

  for i in tqdm(range(data.shape[0])):
    # features
    feat = data.iloc[i, 9:-1]

    # label
    y = np.array(float(data.loc[i]["ee"]) / 100)

    feat_set.append(feat)
    y_set.append(y)

  # split of train & test set
  ratio = 0.8

  # rxnfp
  batch = len(y_set)
  trainset = [feat_set[0: int(ratio * batch)], y_set[0: int(ratio * batch)]]
  testset = [feat_set[int(ratio * batch) + 1:], y_set[int(ratio * batch) + 1:]]


  # report
  dir_path = "%s/exp/Heck_JCP/Benchmark_rs=%s_%s" % (doc_name, rs, datetime.datetime.now())
  os.mkdir("%s" % dir_path)
  f = open("%s/Model_Training_Report.txt" % dir_path, mode="w")

  # record
  f.write("params:\n")
  f.write("random_state=%s\n" % random_state)
  f.write("ratio=%s\n" % ratio)
  f.write("\n")

  # 3.Machine Learning methods
  # RandomForest
  from sklearn.ensemble import RandomForestRegressor

  # Train
  print("RandomForest Start Training")
  start = time.time()

  rf = RandomForestRegressor(bootstrap=True, criterion='squared_error', max_depth=10000,
                                     max_features='sqrt', max_leaf_nodes=None,
                                     min_impurity_decrease=0.0,
                                     min_samples_leaf=1, min_samples_split=2,
                                     min_weight_fraction_leaf=0.0, n_estimators=200, n_jobs=1,
                                     oob_score=False, random_state=0, verbose=0, warm_start=False)
  rf.fit(trainset[0], trainset[1])

  print("Finish Training")
  print("Training Time: %.2f s" % (time.time() - start))  # 截止时间

  # Eval
  # trainset
  # R2
  train_R2 = R2(rf.predict(trainset[0]), np.array(trainset[1]))

  # RMSE
  train_RMSE = RMSE(rf.predict(trainset[0]), np.array(trainset[1]))

  # MAE
  train_MAE = MAE(rf.predict(trainset[0]), np.array(trainset[1]))


  # R2
  test_R2 = R2(rf.predict(testset[0]), np.array(testset[1]))

  # RMSE
  test_RMSE = RMSE(rf.predict(testset[0]), np.array(testset[1]))

  # MAE
  test_MAE = MAE(rf.predict(testset[0]), np.array(testset[1]))


  # Record
  f.write("RandomForest:\n")
  f.write("params:\n")
  f.write("RF:%s\n" % rf.n_estimators)

  f.write("train_R2=%s\n" % train_R2)
  f.write("train_RMSE=%s\n" % train_RMSE)
  f.write("train_MAE=%s\n" % train_MAE)
  f.write("test_R2=%s\n" % test_R2)
  f.write("test_RMSE=%s\n" % test_RMSE)
  f.write("test_MAE=%s\n" % test_MAE)
  f.write("\n")

  eval_metrics.loc["%s" % rs]["train_R2"] = train_R2
  eval_metrics.loc["%s" % rs]["train_RMSE"] = train_RMSE
  eval_metrics.loc["%s" % rs]["train_MAE"] = train_MAE
  eval_metrics.loc["%s" % rs]["test_R2"] = test_R2
  eval_metrics.loc["%s" % rs]["test_RMSE"] = test_RMSE
  eval_metrics.loc["%s" % rs]["test_MAE"] = test_MAE

  # RF Figure
  # RXNFP
  fig = plt.figure(dpi=300, figsize=(10, 5))
  # Test set performance
  tr = np.array(testset[1]).flatten() * 100
  pr = rf.predict(testset[0]).flatten() * 100
  plt.scatter(pr, tr, alpha=0.7, marker=".")
  plt.xlabel("Predicted Yield", fontsize=10)
  plt.ylabel("Observed Yield", fontsize=10)
  x = np.linspace(0, 100, 100)
  y = np.linspace(0, 100, 100)
  plt.plot(x, y, linestyle="--", color="r")
  plt.title("Test set performance", fontsize=15)

  f.close()


In [ ]:
# Evaluation metrics report
for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
  for j in range(eval_metrics.shape[1]):
    eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
eval_metrics.to_csv("%s/exp/Heck_JCP/Benchmark_report_%s.csv" % (doc_name, datetime.datetime.now()))
print(eval_metrics)